# 06 · Sensitivity: do the results depend on our numbers?

Many numbers in notebook 02 are grade D (our assumptions) or C (a conversion), and the effects taken from the studies
are pooled across different scales. This notebook changes the most influential ones and **reruns the actual
pipeline** (notebooks 02 → 05, unchanged code) on each variant, then compares the outcomes with the base run:

- does the data still follow the studies' directions (notebook 02's checks)?
- how many well-defined profiles are there, and which (k, depth) does notebook 05's rule choose?
- does the questionnaire (profiling plus follow-ups, 15 questions) still beat one preset for everyone, and by how much?

How it works: for each variant, notebooks 02–05 run with a few edits made on the fly: the data folder points to
`data/sensitivity/<variant>/`, notebook 02's parameters get the variant's change right after they are loaded, the
grids in 03–05 are trimmed so a variant runs in minutes, and `decisions.json` does not bind (each variant gets the
rule's own choice). Failed checks inside a notebook are recorded, not fatal. Large files are deleted once a variant's
measures are taken.

In [1]:
import json, re, shutil, time
from pathlib import Path
import numpy as np, pandas as pd
import nbformat
from nbclient import NotebookClient
from sklearn.decomposition import PCA

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
NOTEBOOKS = ROOT / "notebooks"
OUT = DATA / "sensitivity"
items = pd.read_csv(DATA / "items.csv")
ORDINAL = np.array([0.0, 1.0, 0.5, 0.5, 0.5])

## Variants

Each is code applied to notebook 02's parameters `P` (read from `parameters.json`) right after they are loaded.

In [2]:
SCALE = 'def scale(entry, f):\n    entry["value"] *= f\n'
SPREADS = ('for key in ["intercept", "data_type", "data_type_x_purpose"]: scale(P["random_sd"][key], {f})\n'
           'for group in ["purpose", "handling_context"]:\n    for e in P["random_sd"][group].values(): scale(e, {f})\n')
EFFECTS = 'for field in ["purpose", "data_type", "handling_context"]:\n    for e in P[field].values(): scale(e, {f})\n'
VARIANTS = {
    "concern weight x0.5": SCALE + 'scale(P["concern_pattern_weight"], 0.5)',
    "concern weight x1.5": SCALE + 'scale(P["concern_pattern_weight"], 1.5)',
    "household spreads x0.5": SCALE + SPREADS.format(f=0.5),
    "household spreads x1.5": SCALE + SPREADS.format(f=1.5),
    "study effects x0.75": SCALE + EFFECTS.format(f=0.75),
    "study effects x1.25": SCALE + EFFECTS.format(f=1.25),
    "no data type x purpose fit": SCALE + 'for e in P["data_type_x_purpose_fit"].values(): scale(e, 0.0)',
}
EDITS = {   # notebook -> (text to find, replacement); the data folder edit is added for every notebook
    "03_clustering.ipynb": [("KS = [1, 2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 24, 32, 40]", "KS = [1, 2, 3, 4, 5, 6, 8, 40]"),
                            ("CANDIDATES = [2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 24]", "CANDIDATES = [2, 3, 4, 5, 6, 8]"),
                            ("BOOTSTRAPS, STABLE, MIN_DIFFERENT = 20, 0.75, 0.10", "BOOTSTRAPS, STABLE, MIN_DIFFERENT = 8, 0.75, 0.10")],
    "04_presets.ipynb": [("RULE_COUNTS = [5, 10, 15, 20, 30, 40, 60, 80, 120, 160]", "RULE_COUNTS = [5, 15, 20, 30, 60]")],
    "05_onboarding_tree.ipynb": [("DEPTHS = list(range(1, 13))", "DEPTHS = list(range(3, 9))"),
                                 ("assert (TEAM_K, TEAM_DEPTH) == (best_k, best_depth), (", "assert True or (TEAM_K, TEAM_DEPTH) == (best_k, best_depth), ("),
                                 ('(ROOT / "reports").mkdir(exist_ok=True)\nfig.savefig(ROOT / "reports" / "onboarding-tree.svg", bbox_inches="tight")',
                                  'fig.savefig(DATA / "onboarding-tree.svg", bbox_inches="tight")')],
}

## Running a variant

In [3]:
def run_notebook(name, data_dir, change=None):
    nb = nbformat.read(NOTEBOOKS / name, as_version=4)
    for cell in nb.cells:
        if cell.cell_type != "code":
            continue
        cell.source = (cell.source.replace('DATA, REF = ROOT / "data", ROOT / "references"', f'DATA, REF = Path(r"{data_dir}"), ROOT / "references"')
                                  .replace('DATA = ROOT / "data"', f'DATA = Path(r"{data_dir}")'))
        for find, replace in EDITS.get(name, []):
            cell.source = cell.source.replace(find, replace)
        if change and 'P = json.loads((DATA / "parameters.json").read_text())' in cell.source:
            cell.source = cell.source.replace('P = json.loads((DATA / "parameters.json").read_text())',
                                              'P = json.loads((DATA / "parameters.json").read_text())\n' + change)
    NotebookClient(nb, timeout=3600, kernel_name="python3", allow_errors=True,
                   resources={"metadata": {"path": str(NOTEBOOKS)}}).execute()
    errors = []
    for c in nb.cells:
        for o in c.get("outputs", []) if c.cell_type == "code" else []:
            if o.output_type == "error":
                failed = [m[1] for m in re.findall(r"""(['"])(.+?)\1: (?:np\.)?False_?""", o.evalue)]   # names of the checks that did not hold
                errors += failed or [f"{o.ename}: {o.evalue[:200]}"]
    return errors

def measures(data_dir):
    d = Path(data_dir)
    D = np.load(d / "decisions.npy")
    V = ORDINAL[D]
    by = lambda f: pd.Series(V.mean(axis=0)).groupby(items[f]).mean()
    quality = pd.read_csv(d / "profiles" / "quality.csv").set_index("k")
    tree = json.loads((d / "onboarding_tree.json").read_text())
    def leaves(n): return [n["preset"]] if "preset" in n else [p for b in n["branches"] for p in leaves(b["next"])]
    return {
        "accepted share": float((D != 0).mean()), "largest pattern (PC1)": float(PCA(1, random_state=0).fit(V[::5]).explained_variance_ratio_[0]),
        "video below sensor data": by("data_type")["ppd:contentData"] < by("data_type")["ppd:sensorData"],
        "ads least accepted": by("purpose").idxmin() == "ppd:advertisingAndMarketing",
        "third party below maker": by("handling_context")["ppd:thirdPartyContext"] < by("handling_context")["ppd:vendorContext"],
        "well-defined k": ", ".join(map(str, quality.index[quality["well defined"]])),
        "chosen k": tree["k"], "chosen depth": tree["depth"], "presets reached": len(set(leaves(tree["root"]))),
        "one preset (all)": tree["one_preset_all_cv"], "after profiling (all)": tree["match_all_cv"],
        "after follow-ups (all)": tree["match_all_after_follow_ups_cv"],
        "gain (points, all)": 100 * (tree["match_all_after_follow_ups_cv"] - tree["one_preset_all_cv"]),
        "gain (points, real-device)": 100 * (tree["match_real_after_follow_ups_cv"] - tree["one_preset_real_cv"]),
    }

## Run all variants (several minutes each)

In [4]:
results, failures = {"base": measures(DATA)}, {"base": []}
for name, change in VARIANTS.items():
    started = time.time()
    d = OUT / re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    shutil.copy(DATA / "items.csv", d / "items.csv"); shutil.copytree(DATA / "corpus", d / "corpus")
    errs = []
    for nb_name in ["02_generate_households.ipynb", "03_clustering.ipynb", "04_presets.ipynb", "05_onboarding_tree.ipynb"]:
        errs += [f"{nb_name[:2]}: {e}" for e in run_notebook(nb_name, d, change if nb_name.startswith("02") else None)]
    try:
        results[name] = measures(d)
    except FileNotFoundError as e:
        results[name] = {}; errs.append(f"measures: {e}")
    failures[name] = errs
    for big in ["decisions.npy", "decisions_cells.npy"]:
        (d / big).unlink(missing_ok=True)
    print(f"{name}: {time.time() - started:.0f} s, {len(errs)} failed checks")
R = pd.DataFrame(results).T

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


concern weight x0.5: 119 s, 0 failed checks


...

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


no data type x purpose fit: 113 s, 2 failed checks


## Results

In [5]:
R[["accepted share", "largest pattern (PC1)", "video below sensor data", "ads least accepted", "third party below maker"]].round(3)

,accepted share,largest pattern (PC1),video below sensor data,ads least accepted,third party below maker
base,0.532167,0.250807,True,True,True
concern weight x0.5,0.532088,0.261504,True,True,True
concern weight x1.5,0.531341,0.244461,True,True,True
household spreads x0.5,0.527132,0.242865,True,True,True
household spreads x1.5,0.533007,0.256915,True,True,True
study effects x0.75,0.528395,0.255484,True,True,True
study effects x1.25,0.534309,0.248491,True,True,True
no data type x purpose fit,0.535661,0.251765,True,True,True


In [6]:
R[["well-defined k", "chosen k", "chosen depth", "presets reached", "one preset (all)", "after profiling (all)",
   "after follow-ups (all)", "gain (points, all)", "gain (points, real-device)"]].round(3)

,well-defined k,chosen k,chosen depth,presets reached,one preset (all),after profiling (all),after follow-ups (all),"gain (points, all)","gain (points, real-device)"
base,"2, 3, 4",4,6,4,0.739,0.786,0.803,6.4,2.1
concern weight x0.5,"2, 3, 4",4,7,4,0.741,0.781,0.798,5.7,2.2
concern weight x1.5,"2, 3, 4",4,5,4,0.736,0.78,0.798,6.2,2.8
household spreads x0.5,"2, 3, 4",4,6,4,0.727,0.774,0.791,6.4,2.3
household spreads x1.5,"2, 3, 4",3,5,3,0.741,0.78,0.798,5.7,1.9
study effects x0.75,"2, 3, 4",4,6,4,0.739,0.781,0.798,5.9,2.0
study effects x1.25,"2, 3, 4",4,6,4,0.736,0.78,0.797,6.1,4.3
no data type x purpose fit,"2, 3, 4",4,6,4,0.735,0.774,0.791,5.6,4.8


In [7]:
pd.Series({k: "; ".join(v) if v else "none" for k, v in failures.items()}, name="checks that failed inside notebooks 02-05")

base                                                                       none
concern weight x0.5                                                        none
concern weight x1.5           03: allow/deny accuracy within S20's 75-90% fo...
household spreads x0.5        04: the chosen rule count keeps most of the pr...
household spreads x1.5                                                     none
study effects x0.75           04: the chosen rule count keeps most of the pr...
study effects x1.25                                                        none
no data type x purpose fit    02: AssertionError: a situation answer tracks ...
Name: checks that failed inside notebooks 02-05, dtype: str

## Checks

In [8]:
checks = {
    "every variant still follows the studies' directions": R[["video below sensor data", "ads least accepted", "third party below maker"]].all().all(),
    "the questionnaire beats one preset for everyone in every variant (all items)": (R["gain (points, all)"] > 0).all(),
    "every variant's onboarding reaches all of its presets": (R["presets reached"] == R["chosen k"]).all(),
}
assert all(checks.values()), checks
pd.Series(checks, name="holds")

every variant still follows the studies' directions                             True
the questionnaire beats one preset for everyone in every variant (all items)    True
every variant's onboarding reaches all of its presets                           True
Name: holds, dtype: bool

## How to read this

- **Directions** follow from the studies' effects, so they should hold in every variant; this is a sanity check.
- **Gain over one preset** (all items, after the follow-ups) is the main outcome. If it stays positive across
  variants, "a short questionnaire beats one preset for everyone" does not depend on our exact numbers. Its size does,
  mostly on how much households differ (the household spreads) and on how decisive the concern types are.
- **Checks that fail inside a variant** are expected at the edges and are listed, not fatal: a variant can push one k
  outside S20's accuracy range, the trimmed rule-count grid (5, 15, 20, 30, 60) can land below the 80% share of the
  profile's match that notebook 04 asks of its full grid, and a near-unanimous situation makes an answer's
  correlation with its own cell unstable. None changes the conclusions above.
- **Well-defined k and the chosen k** show whether the *number* of presets is a property of the data or of our
  assumptions. If it moves with the spreads or the concern weight, the number of presets should be re-derived (notebooks
  03–05, same rules) once real answers are available, before presets reach households.